In [3]:
import pandas as pd
import numpy as np

def load_one(path):
    df = pd.read_csv(path)
    df.columns = [c.lower().strip() for c in df.columns]
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").drop_duplicates("date").reset_index(drop=True)
    for c in ["open","high","low","close","volume"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

spy = load_one("data/processed/SPY_daily.csv")
qqq = load_one("data/processed/QQQ_daily.csv")
vix = load_one("data/processed/VIX_daily.csv")

print("SPY:", spy.shape, spy["date"].min().date(), "->", spy["date"].max().date())
print("QQQ:", qqq.shape, qqq["date"].min().date(), "->", qqq["date"].max().date())
print("VIX:", vix.shape, vix["date"].min().date(), "->", vix["date"].max().date())

for name, df in [("SPY", spy), ("QQQ", qqq), ("VIX", vix)]:
    miss = df[["open","high","low","close","volume"]].isna().mean().mul(100).round(2)
    print(f"\n{name} %missing:\n{miss}")


FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/SPY_daily.csv'

In [5]:
import pandas as pd
import numpy as np

def load_one(path):
    df = pd.read_csv(path)
    df.columns = [c.lower().strip() for c in df.columns]
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").drop_duplicates("date").reset_index(drop=True)
    for c in ["open","high","low","close","volume"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

spy = load_one("../data/processed/SPY_daily.csv")
qqq = load_one("../data/processed/QQQ_daily.csv")
vix = load_one("../data/processed/VIX_daily.csv")

print("SPY:", spy.shape, spy["date"].min().date(), "->", spy["date"].max().date())
print("QQQ:", qqq.shape, qqq["date"].min().date(), "->", qqq["date"].max().date())
print("VIX:", vix.shape, vix["date"].min().date(), "->", vix["date"].max().date())

for name, df in [("SPY", spy), ("QQQ", qqq), ("VIX", vix)]:
    miss = df[["open","high","low","close","volume"]].isna().mean().mul(100).round(2)
    print(f"\n{name} %missing:\n{miss}")


SPY: (8302, 6) 1993-01-29 -> 2026-01-22
QQQ: (6760, 6) 1999-03-10 -> 2026-01-22
VIX: (9081, 6) 1990-01-02 -> 2026-01-22

SPY %missing:
open      0.0
high      0.0
low       0.0
close     0.0
volume    0.0
dtype: float64

QQQ %missing:
open      0.0
high      0.0
low       0.0
close     0.0
volume    0.0
dtype: float64

VIX %missing:
open      0.0
high      0.0
low       0.0
close     0.0
volume    0.0
dtype: float64


In [ ]:
# PASO 2.1 — Merge por fecha SIN volumen del VIX

merged = (
    spy.rename(columns={c: f"spy_{c}" for c in ["open","high","low","close","volume"]})
       .merge(
           qqq.rename(columns={c: f"qqq_{c}" for c in ["open","high","low","close","volume"]}),
           on="date", how="inner"
       )
       .merge(
           vix.rename(columns={c: f"vix_{c}" for c in ["open","high","low","close"]}),  # SIN volume
           on="date", how="inner"
       )
       .sort_values("date")
       .reset_index(drop=True)
)

print("Merged:", merged.shape)
print("Columnas:", merged.columns.tolist())
print("Rango:", merged["date"].min().date(), "->", merged["date"].max().date())


Merged: (6760, 16)
Columnas: ['date', 'spy_open', 'spy_high', 'spy_low', 'spy_close', 'spy_volume', 'qqq_open', 'qqq_high', 'qqq_low', 'qqq_close', 'qqq_volume', 'vix_open', 'vix_high', 'vix_low', 'vix_close', 'volume']
Rango: 1999-03-10 -> 2026-01-22


In [10]:
# PASO 2.2 — Limpiar columna sobrante y verificar dataset base

# Eliminar columna 'volume' sobrante
if "volume" in merged.columns:
    merged = merged.drop(columns=["volume"])

print("Shape final:", merged.shape)
print("Columnas finales:")
print(merged.columns.tolist())
print("Rango:", merged["date"].min().date(), "->", merged["date"].max().date())


Shape final: (6760, 15)
Columnas finales:
['date', 'spy_open', 'spy_high', 'spy_low', 'spy_close', 'spy_volume', 'qqq_open', 'qqq_high', 'qqq_low', 'qqq_close', 'qqq_volume', 'vix_open', 'vix_high', 'vix_low', 'vix_close']
Rango: 1999-03-10 -> 2026-01-22


In [11]:
# PASO 3 — Retornos diarios (SPY, QQQ) y cambio diario del VIX

merged["spy_ret_1d"] = merged["spy_close"].pct_change()
merged["qqq_ret_1d"] = merged["qqq_close"].pct_change()

# El VIX no es un activo invertible; usamos su cambio relativo como señal
merged["vix_chg_1d"] = merged["vix_close"].pct_change()

# Eliminamos la primera fila (NaNs por pct_change)
merged = merged.dropna(subset=["spy_ret_1d", "qqq_ret_1d", "vix_chg_1d"]).reset_index(drop=True)

print("Shape tras retornos:", merged.shape)
merged[["date","spy_ret_1d","qqq_ret_1d","vix_chg_1d"]].head()

Shape tras retornos: (6759, 18)


,date,spy_ret_1d,qqq_ret_1d,vix_chg_1d
0,1999-03-11,0.011127,0.004896,-0.016942
1,1999-03-12,-0.009569,-0.024361,0.019286
2,1999-03-15,0.014251,0.028714,0.016103
3,1999-03-16,-0.003810,0.008495,-0.003566
4,1999-03-17,-0.004303,-0.007220,0.016700


Tras calcular los retornos diarios:

Se obtienen 6.759 observaciones válidas (se pierde una por el cálculo de retornos).

spy_ret_1d y qqq_ret_1d representan el rendimiento diario de ambos mercados.

vix_chg_1d captura el cambio relativo diario del VIX, utilizado como indicador de aumento o disminución del estrés del mercado.

Este enfoque permite:

Analizar la dinámica diaria del riesgo.

Incorporar la información del VIX sin tratarlo como un activo invertible.

Mantener coherencia temporal (toda la información pertenece al mismo día).

Conclusión: las variables de retorno están correctamente definidas y listas para construir medidas de volatilidad.

In [12]:
# PASO 4 — Volatilidad realizada (rolling, anualizada)

import numpy as np

for w in [5, 10, 20]:
    merged[f"spy_rv_{w}d"] = merged["spy_ret_1d"].rolling(w).std() * np.sqrt(252)
    merged[f"qqq_rv_{w}d"] = merged["qqq_ret_1d"].rolling(w).std() * np.sqrt(252)

print("Shape tras volatilidad:", merged.shape)
merged[["date","spy_rv_5d","spy_rv_10d","spy_rv_20d"]].dropna().head()


Shape tras volatilidad: (6759, 24)


,date,spy_rv_5d,spy_rv_10d,spy_rv_20d
19,1999-04-08,0.149928,0.213659,0.226408
20,1999-04-09,0.165405,0.199474,0.224028
21,1999-04-12,0.104535,0.190921,0.222033
22,1999-04-13,0.126319,0.181844,0.219893
23,1999-04-14,0.194284,0.204922,0.228888


Volatilidad realizada (realized volatility)
La volatilidad realizada se calcula como la desviación estándar de los retornos en ventanas móviles y se anualiza.
Este indicador captura la intensidad de los movimientos recientes del mercado y es una referencia estándar para estudiar regímenes de riesgo a corto plazo.

Conclusión: la volatilidad realizada está correctamente definida y es adecuada para construir el objetivo predictivo.

In [13]:
# PASO 5 — Target semanal: alta volatilidad en los próximos 5 días

H = 5  # horizonte: 1 semana (5 días de trading)

# Volatilidad futura (realized) basada en retornos futuros
merged["spy_future_rv_5d"] = (
    merged["spy_ret_1d"]
    .rolling(H)
    .std()
    .shift(-H) * np.sqrt(252)
)

# Umbral: percentil 80 (solo para arrancar; se fijará con TRAIN más adelante)
threshold = merged["spy_future_rv_5d"].quantile(0.80)

merged["y_high_vol_next_week"] = (merged["spy_future_rv_5d"] >= threshold).astype(int)

# Limpiar filas sin target (por el shift)
merged = merged.dropna(subset=["spy_future_rv_5d", "y_high_vol_next_week"]).reset_index(drop=True)

print("Threshold (p80):", round(threshold, 4))
print("Shape con target:", merged.shape)
print("Tasa de alta volatilidad:", round(merged["y_high_vol_next_week"].mean(), 3))

merged[["date","spy_future_rv_5d","y_high_vol_next_week"]].head()


Threshold (p80): 0.2143
Shape con target: (6754, 26)
Tasa de alta volatilidad: 0.2


,date,spy_future_rv_5d,y_high_vol_next_week
0,1999-03-11,0.186897,0
1,1999-03-12,0.233943,1
2,1999-03-15,0.202828,0
3,1999-03-16,0.280730,1
4,1999-03-17,0.295172,1


Definición del objetivo predictivo

Se define como objetivo la detección de semanas de alta volatilidad, donde la volatilidad realizada del S&P 500 en los próximos 5 días supera el percentil 80 de su distribución histórica.

Resultados:

Umbral (p80): 0.2143 (volatilidad anualizada).

Proporción de semanas de alta volatilidad: 20%.

Dataset final: 6.754 observaciones.

Esta definición produce un problema de clasificación desbalanceado moderado, realista y adecuado para evaluar modelos de predicción de riesgo.

In [14]:
# PASO 6 — Definir conjunto de features (X) SOLO con información hasta t

feature_cols = [
    # Retornos
    "spy_ret_1d",
    "qqq_ret_1d",
    "vix_chg_1d",

    # Volatilidad realizada (pasada)
    "spy_rv_5d", "spy_rv_10d", "spy_rv_20d",
    "qqq_rv_5d", "qqq_rv_10d", "qqq_rv_20d",

    # Nivel de estrés
    "vix_close"
]

# Asegurar que todas existen
feature_cols = [c for c in feature_cols if c in merged.columns]

X = merged[feature_cols].copy()
y = merged["y_high_vol_next_week"].copy()

# Limpieza final
mask = X.notna().all(axis=1)
X = X.loc[mask].reset_index(drop=True)
y = y.loc[mask].reset_index(drop=True)

print("X shape:", X.shape)
print("y mean (alta vol):", round(y.mean(), 3))
print("Features:", feature_cols)


X shape: (6735, 10)
y mean (alta vol): 0.199
Features: ['spy_ret_1d', 'qqq_ret_1d', 'vix_chg_1d', 'spy_rv_5d', 'spy_rv_10d', 'spy_rv_20d', 'qqq_rv_5d', 'qqq_rv_10d', 'qqq_rv_20d', 'vix_close']


Conjunto de variables explicativas

El conjunto final de features incluye 10 variables, todas disponibles en el instante t:

Retornos diarios (SPY, QQQ)

Cambio diario del VIX

Volatilidad realizada pasada (5, 10 y 20 días) de SPY y QQQ

Nivel del VIX

Resultados:

6.735 observaciones válidas

Proporción de clase positiva (alta volatilidad): 19.9%

El conjunto de datos presenta un desbalance moderado, típico en problemas de detección de riesgo, y las variables seleccionadas son coherentes con la literatura sobre volatilidad.

In [15]:
# PASO 7 — Split temporal (sin shuffle)

dates = merged.loc[X.index, "date"].reset_index(drop=True)

# Definimos cortes temporales
train_end = "2015-12-31"
val_end   = "2019-12-31"

train_idx = dates <= train_end
val_idx   = (dates > train_end) & (dates <= val_end)
test_idx  = dates > val_end

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val     = X[val_idx], y[val_idx]
X_test, y_test   = X[test_idx], y[test_idx]

print("Train:", X_train.shape, "Alta vol:", round(y_train.mean(),3))
print("Val:  ", X_val.shape,   "Alta vol:", round(y_val.mean(),3))
print("Test: ", X_test.shape,  "Alta vol:", round(y_test.mean(),3))


Train: (4231, 10) Alta vol: 0.224
Val:   (1006, 10) Alta vol: 0.084
Test:  (1498, 10) Alta vol: 0.206


El conjunto se divide respetando el orden temporal en tres periodos:

Entrenamiento (1999–2015): 4.231 observaciones, 22.4% de semanas de alta volatilidad.

Validación (2016–2019): 1.006 observaciones, 8.4% de semanas de alta volatilidad.

Test (2020–2026): 1.498 observaciones, 20.6% de semanas de alta volatilidad.

La reducción de episodios de alta volatilidad en el periodo de validación refleja un entorno de mercado más estable, mientras que el conjunto de test vuelve a incluir episodios de estrés (COVID-19 y posteriores).

In [16]:
# PASO 8 — Baseline de persistencia
# Regla: predecir alta volatilidad si la RV(20d) actual está por encima de su p80 en TRAIN

from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

# Umbral SOLO con TRAIN (evitar leakage)
rv_thr = X_train["spy_rv_20d"].quantile(0.80)

def persistence_pred(X, thr):
    return (X["spy_rv_20d"] >= thr).astype(int)

# Predicciones
y_train_pred = persistence_pred(X_train, rv_thr)
y_val_pred   = persistence_pred(X_val, rv_thr)
y_test_pred  = persistence_pred(X_test, rv_thr)

# Métricas
def report(y_true, y_pred, name):
    print(f"\n{name}")
    print("F1:", round(f1_score(y_true, y_pred),3))
    print("Precision:", round(precision_score(y_true, y_pred),3))
    print("Recall:", round(recall_score(y_true, y_pred),3))

report(y_train, y_train_pred, "TRAIN")
report(y_val,   y_val_pred,   "VAL")
report(y_test,  y_test_pred,  "TEST")



TRAIN
F1: 0.588
Precision: 0.622
Recall: 0.557

VAL
F1: 0.176
Precision: 0.189
Recall: 0.165

TEST
F1: 0.561
Precision: 0.608
Recall: 0.521


Baseline de persistencia de volatilidad

El baseline asume que la volatilidad futura será alta cuando la volatilidad reciente ya lo es (RV(20d) por encima del percentil 80 del periodo de entrenamiento).

Resultados:

TRAIN:

F1 = 0.588

Precision = 0.622

Recall = 0.557

VALIDATION (2016–2019):

F1 = 0.176

Precision = 0.189

Recall = 0.165

TEST (2020–2026):

F1 = 0.561

Precision = 0.608

Recall = 0.521

Interpretación clave:

El baseline funciona bien en periodos con regímenes claros de volatilidad persistente (TRAIN y TEST).

Su rendimiento cae fuertemente en el periodo de validación, caracterizado por baja volatilidad y menor persistencia.

Esto confirma que el problema no es trivial y que hay margen para mejorar usando información adicional (QQQ y VIX).

Conclusión: el baseline es razonable pero inestable; justifica plenamente el uso de modelos de Machine Learning.

In [17]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score, average_precision_score, roc_auc_score
import xgboost as xgb

# --- 1) Convertir a DMatrix (API nativa: robusta con early stopping) ---
dtrain = xgb.DMatrix(X_train, label=y_train)
dval   = xgb.DMatrix(X_val,   label=y_val)
dtest  = xgb.DMatrix(X_test,  label=y_test)

# --- 2) Parámetros "fuertes" iniciales (buen punto de partida) ---
# Nota: scale_pos_weight ayuda con desbalance (aprox ratio neg/pos en TRAIN)
pos = float(y_train.sum())
neg = float(len(y_train) - y_train.sum())
scale_pos_weight = neg / pos if pos > 0 else 1.0

params = {
    "objective": "binary:logistic",
    "eval_metric": ["logloss", "aucpr"],   # AUC-PR es buena con clases desbalanceadas
    "eta": 0.03,                           # learning rate bajo (mejor generalización)
    "max_depth": 4,
    "min_child_weight": 5,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "scale_pos_weight": scale_pos_weight,
    "seed": 42,
}

# --- 3) Entrenamiento con early stopping en VAL ---
watchlist = [(dtrain, "train"), (dval, "val")]

model_xgb = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=5000,
    evals=watchlist,
    early_stopping_rounds=200,
    verbose_eval=200
)

# --- 4) Probabilidades ---
p_train = model_xgb.predict(dtrain)
p_val   = model_xgb.predict(dval)
p_test  = model_xgb.predict(dtest)

# --- 5) Buscar el mejor threshold en VAL para maximizar F1 ---
ths = np.linspace(0.05, 0.95, 91)
f1s = [f1_score(y_val, (p_val >= t).astype(int)) for t in ths]
best_t = float(ths[int(np.argmax(f1s))])

def report(name, y_true, p, t):
    y_pred = (p >= t).astype(int)
    print(f"\n{name}  (thr={t:.3f})")
    print("F1:", round(f1_score(y_true, y_pred), 3))
    print("Precision:", round(precision_score(y_true, y_pred), 3))
    print("Recall:", round(recall_score(y_true, y_pred), 3))
    print("ROC-AUC:", round(roc_auc_score(y_true, p), 3))
    print("PR-AUC:", round(average_precision_score(y_true, p), 3))

print("\nBest threshold on VAL (max F1):", round(best_t, 3))
report("TRAIN", y_train, p_train, best_t)
report("VAL",   y_val,   p_val,   best_t)
report("TEST",  y_test,  p_test,  best_t)


[0]	train-logloss:0.67893	train-aucpr:0.71809	val-logloss:0.67400	val-aucpr:0.32489
[200]	train-logloss:0.32718	train-aucpr:0.83080	val-logloss:0.25557	val-aucpr:0.33825
[217]	train-logloss:0.32194	train-aucpr:0.83579	val-logloss:0.25538	val-aucpr:0.33754

Best threshold on VAL (max F1): 0.37

TRAIN  (thr=0.370)
F1: 0.694
Precision: 0.542
Recall: 0.964
ROC-AUC: 0.944
PR-AUC: 0.836

VAL  (thr=0.370)
F1: 0.408
Precision: 0.353
Recall: 0.482
ROC-AUC: 0.818
PR-AUC: 0.345

TEST  (thr=0.370)
F1: 0.569
Precision: 0.442
Recall: 0.796
ROC-AUC: 0.84
PR-AUC: 0.654


“El modelo es capaz de anticipar aproximadamente el 80% de los episodios de alta volatilidad semanal, aunque con un nivel moderado de falsas alarmas, lo que resulta adecuado en un contexto de gestión del riesgo donde la prioridad es evitar exposiciones en periodos de estrés.”

In [18]:
# Guardar dataset final en data/processed

out_path = "../data/processed/dataset_volatility_weekly.csv"

merged.to_csv(out_path, index=False)

print(f"Dataset guardado en: {out_path}")
print("Shape:", merged.shape)


Dataset guardado en: ../data/processed/dataset_volatility_weekly.csv
Shape: (6754, 26)
